# Варіант 1. Пошук службових повідомлень за змістом
**Тема 4, заняття 15 · Індивідуальний проєкт · 40 хв.**

**Мета:** побудувати невеликий пошуковий модуль з TF–IDF і косинусною схожістю.

**Матеріал заняття 4.14:** очищення тексту → векторизація TF–IDF → пошук схожих документів → перевірка релевантності.

**Готовий результат:** 3 найрелевантніші повідомлення з ID та оцінкою схожості; власний висновок про обмеження методу.

**План (40 хв):** 5 — постановка задачі; 10 — завантаження та запуск; 15 — три запити й аналіз результатів; 10 — оформлення висновку.

Усі тексти синтетичні та не містять відомостей обмеженого доступу.

In [1]:
from pathlib import Path
import csv

DATA = Path("Варіант_1_повідомлення_30.csv")
if not DATA.exists():
    DATA = Path("../Варіант_1_повідомлення_30.csv")
with DATA.open(encoding="utf-8-sig", newline="") as f:
    docs = list(csv.DictReader(f))
print("Завантажено повідомлень:", len(docs))
for row in docs[:5]:
    print(row["id"], row["текст"])

Завантажено повідомлень: 30
M001 До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002 Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M003 Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.
M004 Просимо забезпечити клас новими гарнітурами для дистанційного навчання.
M005 Уточнено потребу у папері та картриджах для друку навчальних матеріалів.


## 1. Перетворюємо тексти на вектори
`fit_transform()` створює словник текстів і матрицю TF–IDF. Не потрібно запускати `fit_transform()` окремо для запиту.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X = vectorizer.fit_transform([d["текст"] for d in docs])
print("Розмір TF–IDF матриці:", X.shape)

Розмір TF–IDF матриці: (30, 385)


## 2. Пишемо пошукову функцію
Косинусна схожість порівнює вектори, **але не є ймовірністю правдивості** повідомлення.

In [3]:
def search(query, top_k=3):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, X)[0]
    selected = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:top_k]
    for i in selected:
        print(f'{docs[i]["id"]}: {scores[i]:.3f} — {docs[i]["текст"]}')

# ЗАВДАННЯ: змініть текст запиту та проаналізуйте результати.
query = "постачання кабелів зв’язку"
search(query, top_k=3)

M001: 0.568 — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.073 — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M003: 0.000 — Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.


## 3. Обов'язкові експерименти
Виконайте **три запити** й запишіть ID трьох знайдених повідомлень для кожного:

1. `постачання кабелів зв’язку`
2. `помилка авторизації`
3. `аналітичний звіт про навчання`

**Додатковий експеримент:** порівняйте результати запитів `помилка авторизації` та `несправність входу`.

Відповідайте на запитання:
- Чи всі три відповіді справді відповідають інформаційній потребі?
- Чи знаходить TF–IDF синонімічні формулювання без спільних слів?
- Чому назва теми в CSV не повинна замінювати перевірку змісту?

In [4]:
for q in ["постачання кабелів зв’язку", "помилка авторизації", "аналітичний звіт про навчання"]:
    print("\nЗапит:", q)
    search(q)


Запит: постачання кабелів зв’язку
M001: 0.568 — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.073 — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M003: 0.000 — Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.

Запит: помилка авторизації
M007: 0.420 — Під час входу до навчальної платформи виникла помилка авторизації користувача.
M001: 0.000 — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.000 — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.

Запит: аналітичний звіт про навчання
M013: 0.558 — Підготовлено аналітичний звіт про результати навчального тижня.
M014: 0.247 — До кафедри подано місячний звіт про проведені практичні заняття.
M004: 0.096 — Просимо забезпечити клас новими гарнітурами для дистанційного навчання.


### Додатковий експеримент: синонімічні формулювання

In [5]:
for q in ["помилка авторизації", "несправність входу", "не вдається увійти в систему"]:
    print("\nЗапит:", q)
    search(q)


Запит: помилка авторизації
M007: 0.420 — Під час входу до навчальної платформи виникла помилка авторизації користувача.
M001: 0.000 — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.000 — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.

Запит: несправність входу
M012: 0.490 — Виникла несправність входу до електронного журналу занять.
M007: 0.115 — Під час входу до навчальної платформи виникла помилка авторизації користувача.
M001: 0.000 — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.

Запит: не вдається увійти в систему
M008: 0.303 — Слухач не може увійти до особистого кабінету через невірний пароль.
M009: 0.138 — Система дистанційного навчання тимчасово не відкриває сторінку тестування.
M011: 0.137 — Після оновлення браузера не працює завантаження навчальних файлів.


### Власний запит без спільних слів із документами

In [6]:
search("поставка проводів")

M001: 0.000 — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.000 — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M003: 0.000 — Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.


## 4. Звіт слухача

**Запит, який я перевіряв(ла):** `постачання кабелів зв’язку`, `помилка авторизації`, `аналітичний звіт про навчання`, а також `несправність входу`, `не вдається увійти в систему` і `поставка проводів`.

**Топ-3 ID документів:**

| Запит | Топ-3 (схожість) |
|---|---|
| постачання кабелів зв’язку | M001 (0.568), M002 (0.073), M003 (0.000) |
| помилка авторизації | M007 (0.420), M001 (0.000), M002 (0.000) |
| аналітичний звіт про навчання | M013 (0.558), M014 (0.247), M004 (0.096) |
| несправність входу | M012 (0.490), M007 (0.115), M028 (0.000) |

**Відповідні документи та невідповідні документи:**
- *Кабелі:* M001 і M002 — релевантні (обидва про постачання кабелів). M003 — нерелевантний: схожість 0.000, він потрапив у топ лише тому, що функція завжди повертає 3 результати.
- *Авторизація:* релевантний лише M007. M001 і M002 мають схожість 0 і не стосуються запиту. Водночас M008 («не може увійти … невірний пароль») і M012 («несправність входу») за змістом релевантні, але не знайдені — немає спільних слів.
- *Звіт:* M013 і M014 — релевантні. M004 (гарнітури) — нерелевантний, збіг лише через спільну основу слова «навчання». Документи M016 («аналітична довідка») і M018 («підсумковий огляд») за змістом релевантні, але програють через інші словоформи.
- *Порівняння «помилка авторизації» / «несправність входу»:* запити описують одну проблему, але дають різних лідерів (M007 і M012). M007 у другому запиті з'являється лише завдяки слову «входу». Отже, TF–IDF **не знаходить синоніми без спільних слів**; запит «поставка проводів» узагалі дає нульову схожість з усіма 30 повідомленнями, хоча M001/M002 про кабелі.

**Чому тема в CSV не замінює перевірку змісту:** колонка `тема_для_перевірки` — це ручна мітка для контролю, пошук її не використовує. Мітка може бути помилковою або надто загальною (M010 «нестабільне підключення до мережі» позначено як «Технічна підтримка», але за змістом близьке і до «Ремонту»), а один документ може стосуватися кількох тем. Релевантність визначається змістом тексту щодо інформаційної потреби, а не назвою категорії.

**Одне обмеження TF–IDF:** метод враховує лише точний збіг слів/біграм. Він не розуміє синонімів («кабелі» ≠ «проводи», «авторизація» ≠ «вхід») і різних словоформ української мови («навчання» ≠ «навчального»). Також косинусна схожість — не ймовірність правильності: функція повертає документи навіть зі схожістю 0.

**Моя пропозиція поліпшення:** 1) відсікати результати зі схожістю нижче порогу (напр. < 0.05); 2) лематизація/стемінг українських слів (pymorphy3) або `TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5))` для стійкості до словоформ; 3) словник синонімів для розширення запиту або семантичні ембедінги (багатомовні моделі sentence-transformers).

**Висновок:** пошуковий модуль на TF–IDF правильно знаходить документ-лідера, коли запит і текст мають спільні слова (M001, M007, M013 — схожість 0.42–0.57), але пропускає синонімічні формулювання та повертає «шумові» документи з нульовою схожістю. Для практичного використання потрібні поріг схожості, нормалізація словоформ і ручна перевірка релевантності.